# 4 · Tables

A node that returns many values makes a table, and a node that runs once per row adds a column to the table it ran over. A `Table` input or output is a `polars.DataFrame`.

In [ ]:
from typing import TypedDict

import polars as pl
from conductor import GraphRecord, NodeRecord, NodeRegistry, Series, Table, node


class Document(TypedDict):
    name: str


@node("documents")
def documents() -> Table[Document]:
    return pl.DataFrame({"name": ["a.pdf", "b.pdf"]})


@node("lines")
def lines(name: str) -> Series[str]:
    return ["first line", "second line"] if name == "a.pdf" else []


@node("shout")
def shout(line: str) -> str:
    return line.upper()


@node("count")
def count(values: Series[str]) -> int:
    return len(values)


@node("sheet")
def sheet(rows: Table) -> Table:
    return rows


registry = NodeRegistry()
registry.add(documents, lines, shout, count, sheet)

## Wiring decides how a node runs

`documents.result.name` is one column of the table. `lines` takes one value, so it runs once per document and makes rows under each. `count` takes a `Series`, so it gets every line under its document. `sheet` takes a `Table`, built from the columns wired into it.

In [ ]:
graph = registry.load(GraphRecord(
    nodes=[
        NodeRecord(id="documents", node="documents"),
        NodeRecord(id="lines", node="lines"),
        NodeRecord(id="shout", node="shout"),
        NodeRecord(id="how_many", node="count"),
        NodeRecord(id="per_line", node="sheet"),
        NodeRecord(id="per_document", node="sheet"),
    ],
    edges=[
        ("documents.result.name", "lines.name"),
        ("lines.result", "shout.line"),
        ("lines.result", "how_many.values"),
        ("documents.result.name", "per_line.rows"),
        ("shout.result", "per_line.rows"),
        ("documents.result.name", "per_document.rows"),
        ("how_many.result", "per_document.rows"),
    ],
))
print(graph.problems)
graph.plan

## Building a table

A `Table` input runs one level above its deepest source. `per_line` runs once per document: a table of that document's lines, its name copied down beside each. `b.pdf` has no lines, so its table has no rows. `per_document` runs once, a row per document, so `b.pdf` shows with a count of 0. Rows of two different tables never meet by themselves: that takes a join on a key, in a node.

In [ ]:
results = graph.run().results
for table in results["per_line"]["result"]:
    print(table)
print(results["per_document"]["result"])

## Blanks

A blank cell reaches a node only if its input says `| None`. Otherwise the node isn't called for that row, its output is blank there, and `run.not_called` says which row and input.

In [ ]:
class Contact(TypedDict):
    email: str


@node("contacts")
def contacts() -> Table[Contact]:
    return pl.DataFrame({"email": ["ida@a.dk", None]}, schema={"email": pl.String})


@node("domain")
def domain(email: str) -> str:
    return email.split("@")[1]


@node("has-email")
def has_email(email: str | None) -> bool:
    return email is not None


registry.add(contacts, domain, has_email)
blanks = registry.load(GraphRecord(
    nodes=[NodeRecord(id="contacts", node="contacts"), NodeRecord(id="domain", node="domain"), NodeRecord(id="has_email", node="has-email")],
    edges=[("contacts.result.email", "domain.email"), ("contacts.result.email", "has_email.email")],
))
run = blanks.run()
print(run.results["domain"]["result"], run.results["has_email"]["result"])
print(run.not_called)